In [ ]:
import requests
from bs4 import BeautifulSoup
import csv

"""commentaire
before starting web scraping we create once an virtuale environnement for this project
after we import the requests environment 
the variable url get take a link of the url of the serveur(which contain the web) that will be scraped
and after it we have the header, it is important in the sense that , it notice to the serveur that
it is not an robot but an requet and give it some information in his host, connection user-Agent the most
important thing in this header because it give it some information on the type of system on the sender computer,
the name of the web acces navigator and also all necessaire information to recognize the computer.
and finally there the variable which receive the request and we will extract the data so only what we need and put it in the csv file.
"""
""" now we want to scrape the data for multiple pages, so we will create a loop to scrape the data for multiple pages and we will change 
the page number in the url for each iteration of the loop, and we will also change the name of the csv file for each iteration of the loop 
to avoid overwriting the previous data, and we will also add a delay between each iteration of the loop to avoid getting blocked by the server. """

#donc il faut sauvegarder de temps en temps.
#et faire la boucle pour recuperer chaque page.

with open("test1.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["name of the repository", "description of the repository", "programation language", "number of stars of the repository", "date of the last update of the repository"])

for page_number in range(1, 100):
    url = "https://github.com/search?q=mental+health+ai&type=repositories&p=" + str(page_number)

    header = {
        "Host": "github.com",
        "connection": "keep-alive",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36",
        "Referrer": "https://www/google/com/",
        "Accept-Encoding": "gzip, deflate, sdch",
        "Accept-Language": "en -US, en; q=0.8"
    }

    page = requests.get(url, headers=header)

    html = BeautifulSoup(page.content, "html.parser") #permet d'analyser le code html de la page web et de le rendre plus facile à manipuler

    #permet de trouver les titres de chaque section
    propre1 = []
    html1 = html.find_all("span", class_="search-match SearchMatchText-module__searchMatchText__n6aQc SearchMatchText-module__truncatedName__X6PYS prc-Text-Text-9mHv3")
    for element in html1:
        propre1.append(element.get_text().strip())

    #permet de trouver les descriptions de chaque section
    propre2 = []
    html2 = html.find_all("span", class_="search-match SearchMatchText-module__searchMatchText__n6aQc prc-Text-Text-9mHv3")
    for element in html2:
        propre2.append(element.get_text().strip()) #permet de supprimer les espaces blancs au début et à la fin du texte de l'élément trouvé. Cela peut être utile pour nettoyer les données extraites de la page web avant de les utiliser ou de les stocker.

    #permet de trouver le langage de programmation de chaque section
    propre3 = []
    html3 = html.find_all("span", attrs={"aria-label": True})
    for element in html3:
        propre3.append(element.get_text().strip())

    #permet de trouver le nombre d'étoiles de chaque section
    # On cherche tous les liens dont l'adresse (href) contient 'stargazers'
    stars_elements = html.find_all("a", href=lambda x: x and "stargazers" in x)
    propre4 = [s.get_text().strip() for s in stars_elements]

    # On cherche les spans qui ont un titre (title) ET qui sont dans le pied de page du résultat
    # les dates contiennent souvent "202" (pour les années 2020, 2021, etc.), donc on filtre pour ne garder que ceux-là
    dates_elements = html.find_all("span", title=True)
    propre5 = [d["title"] for d in dates_elements if "202" in d["title"]]

    data = {
        "name of the repository": propre1,
        "description of the repository": propre2,
        "programation language": propre3,
        "number of stars of the repository": propre4,
        "date of the last update of the repository": propre5
    }

    print(data)

    #permet d'écrire les données extraites dans un fichier csv les deux premiere colonnes
    with open("test1.csv", "a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["name of the repository", "description of the repository", "programation language", "number of stars of the repository", "date of the last update of the repository"])
        for name, description, language, stars, updated_at in zip(propre1, propre2, propre3, propre4, propre5):
            writer.writerow({
                "name of the repository": name,
                "description of the repository": description,
                "programation language": language,
                "number of stars of the repository": stars,
                "date of the last update of the repository": updated_at
            })

{'name of the repository': ['kairess/mental-health-chatbot', 'jahnavik186/AI-Mental-Health-Companion', 'PoyBoi/MindEase', 'Shivam1337/MindCrafter', 'RCTS-IIITH/AI-Assisted-Mental-Health-Screening-Application', 'dhrumilp12/Mental-Health-Companion', 'mujiyantosvc/Facial-Expression-Recognition-FER-for-Mental-Health-Detection-', 'SherazAsghar37/ai_mental_health', 'galihru/MentalHealth', 'Sanjana-Rajagopala/AI_Healthcare_Chatbot'], 'description of the repository': ['심리상담 정신건강 상담 챗봇. AI chatbot for psychology consultation.', 'An intelligent, interactive mental health companion that uses Large Language Models (LLMs) combined with RAG to provide context-aware mental', 'AI based Mental Health Counsellor / Mental Health Assistant', 'AI & ML Based Mental Health Diagnosis & Consulting App.', 'To expand the scope of existing mental health screening questionnaires by creating a Chat Bot styled testing application, allowing for en…', 'This app would provide mental health support through conversationa

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import csv
import json
import sys
""" in this part we will scrape the product hunt website to get the products related to mental health ai, we will use selenium to navigate through 
the pages and get the html content of each page, then we will use beautifulsoup to parse the html content and extract the data we need, and finally 
we will save the data in a json file and a csv file. """

# Configuration
START_PAGE = 1
MAX_PAGES = 24
MAX_EMPTY_PAGES = 2
HEADLESS = False
OUTPUT_JSON = "producthunt_results.json"
OUTPUT_CSV = "producthunt_results.csv"

#ce qui permt que l'execution ne fait plus 
# Selenium options
options = Options()
options.add_argument("--width=1200")
options.add_argument("--height=900")
if HEADLESS:
    options.add_argument("-headless")

# Function to parse products from HTML content
def parse_products_from_html(html):
    soup = BeautifulSoup(html, "html.parser")
    products = []

    buttons = soup.select("button[data-test^='spotlight-result-product-']")
    for button in buttons:
        try:
            name_tag = button.select_one("span.text-base.font-semibold.text-dark-gray") or button.select_one("h3")
            tagline_tag = button.select_one("span.text-sm.font-normal.text-light-gray") or button.select_one("p")
            votes_tag = button.select_one("span.text-sm.font-semibold.text-brand-500")

            name = name_tag.get_text(strip=True) if name_tag else None
            tagline = tagline_tag.get_text(strip=True) if tagline_tag else ""
            votes = votes_tag.get_text(strip=True) if votes_tag else "0"

            if name:
                products.append(
                    {
                        "name": name,
                        "tagline": tagline,
                        "votes": votes,
                    }
                )
        except Exception:
            continue
# Fallback parsing if no products found with the main selectors
    if not products:
        title_spans = soup.find_all("span", class_=lambda c: c and "font-semibold" in c)
        for span in title_spans:
            name = span.get_text(strip=True)
            if not name:
                continue
            products.append(
                {
                    "name": name,
                    "tagline": "",
                    "votes": "0",
                }
            )


    return products

# Function to save products incrementally to JSON and CSV files
def save_incremental(all_products, output_file=OUTPUT_JSON):
    json_file = "result_scrap_selenium1.json"
    with open(json_file, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)

    csv_file = "result_scrap_selenium1.csv"
    if all_products:
        with open(csv_file, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=all_products[0].keys())
            writer.writeheader()
            writer.writerows(all_products)

# Function to create a Selenium WebDriver instance
def create_driver():
    return webdriver.Firefox(options=options)

# Main scraping function with retry logic and incremental saving
def scrape(start_page=START_PAGE, max_pages=MAX_PAGES, max_empty_pages=MAX_EMPTY_PAGES):
    all_products = []
    seen_names = set()

    try:
        page_number = start_page
        empty_pages = 0

        while page_number < start_page + max_pages and empty_pages < max_empty_pages:
            search_url = f"https://www.producthunt.com/search?q=mental+health+ai&page={page_number}"
            print(f"Scraping page {page_number} -> {search_url}")

            attempt = 0
            page_html = ""
            page_had_products = False
            driver = None

            while attempt < 4:
                attempt += 1
                try:
                    driver = create_driver()
                    wait = WebDriverWait(driver, 15)
                    driver.get(search_url)
                    time.sleep(4 + attempt)
                    page_html = driver.page_source

#ce bloc est pour eviter les erreurs de captcha
                    blocked_markers = [
                        "Just a moment",
                        "Checking your browser",
                        "Access denied",
                        "Attention Required",
                    ]
                    if any(marker.lower() in page_html.lower() for marker in blocked_markers):
                        print(f"Page {page_number} looks blocked (attempt {attempt}). Retrying...")
                        time.sleep(5 * attempt)
                        continue

                    products = parse_products_from_html(page_html)
                    if products:
                        page_had_products = True
                        for product in products:
                            key = product.get("name")
                            if key in seen_names:
                                continue
                            seen_names.add(key)
                            product["id"] = len(all_products) + 1
                            all_products.append(product)

                        print(f"Found {len(products)} products on page {page_number}")
                        break

                    print(f"No products parsed on page {page_number} (attempt {attempt}).")
                    time.sleep(2)

                except Exception as error:
                    print(f"Driver error on page {page_number} attempt {attempt}: {type(error).__name__}: {error}")
                    time.sleep(3 * attempt)
                finally:
                    if driver is not None:
                        try:
                            driver.quit()
                        except Exception:
                            pass
                        driver = None

            if page_had_products:
                empty_pages = 0
            else:
                empty_pages += 1
                print(f"Empty pages in a row: {empty_pages}/{max_empty_pages}")

            save_incremental(all_products, output_file="results2.json")
            page_number += 1

    except Exception as error:
        print("Fatal error during scraping:", type(error).__name__, error)
    finally:
        if driver is not None:
            try:
                driver.quit()
            except Exception as error:
                print("Error quitting driver:", error)

# Final save after scraping
    save_incremental(all_products, output_file="results2.json")

# Entry point
if __name__ == "__main__":
    max_pages = MAX_PAGES
    if len(sys.argv) >= 2:
        try:
            max_pages = int(sys.argv[1])
        except Exception:
            max_pages = MAX_PAGES

    scrape(start_page=START_PAGE, max_pages=max_pages)




Scraping page 1 -> https://www.producthunt.com/search?q=mental+health+ai&page=1
Found 10 products on page 1
Scraping page 2 -> https://www.producthunt.com/search?q=mental+health+ai&page=2
Found 10 products on page 2
Scraping page 3 -> https://www.producthunt.com/search?q=mental+health+ai&page=3
Found 10 products on page 3
Scraping page 4 -> https://www.producthunt.com/search?q=mental+health+ai&page=4
Found 10 products on page 4
Scraping page 5 -> https://www.producthunt.com/search?q=mental+health+ai&page=5
Found 10 products on page 5
Scraping page 6 -> https://www.producthunt.com/search?q=mental+health+ai&page=6
Found 10 products on page 6
Scraping page 7 -> https://www.producthunt.com/search?q=mental+health+ai&page=7
Found 10 products on page 7
Scraping page 8 -> https://www.producthunt.com/search?q=mental+health+ai&page=8
Found 10 products on page 8
Scraping page 9 -> https://www.producthunt.com/search?q=mental+health+ai&page=9
Found 10 products on page 9
Scraping page 10 -> https://

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import time
import csv
import json
import sys
import re
""" Scraper Product Hunt (mental health AI) avec navigation vers la page de chaque produit.
Chaque produit suit les etapes: BROWSE -> WAIT -> EXTRACT -> RETURN.
(Amélioré : utilise de nouveaux onglets pour éviter de recharger la page de recherche)
"""
# Configuration
START_PAGE = 1
MAX_PAGES = 24
MAX_EMPTY_PAGES = 2
HEADLESS = False
OUTPUT_JSON = "producthunt_results.json"
OUTPUT_CSV = "producthunt_results.csv"
BASE_URL = "https://www.producthunt.com"
# Selenium options
options = Options()
options.add_argument("--width=1200")
options.add_argument("--height=900")
if HEADLESS:
    options.add_argument("-headless")
def absolute_url(href):
    if not href:
        return ""
    if href.startswith("http"):
        return href
    return BASE_URL + href
def parse_products_from_html(html):
    soup = BeautifulSoup(html, "html.parser")
    products = []
    buttons = soup.select("button[data-test^='spotlight-result-product-']")
    for button in buttons:
        try:
            name_tag = button.select_one("span.text-base") or button.select_one("h3") or button.select_one("h2")
            tagline_tag = button.select_one("span.text-sm") or button.select_one("p")
            votes_tag = button.select_one("span.text-sm.font-semibold.text-brand-500")
            link_tag = button.select_one("a[href*='/products/']") or button.select_one("a[href*='/posts/']")
            name = name_tag.get_text(strip=True) if name_tag else None
            tagline = tagline_tag.get_text(strip=True) if tagline_tag else ""
            votes = votes_tag.get_text(strip=True) if votes_tag else "0"
            product_url = absolute_url(link_tag.get("href")) if link_tag and link_tag.has_attr("href") else ""
            if name:
                products.append({
                    "name": name,
                    "tagline": tagline,
                    "votes": votes,
                    "product_url": product_url,
                })
        except Exception:
            continue
    return products
def empty_detail_fields():
    return {
        "description": "",
        "description_detaillee": "",
        "website": "",
        "makers": "",
        "topics": "",
        "reviews": "",
        "app_type": "",
        "developers": "",
        "activity_types": "",
        "id": "",
    }


def extract_detail_fields(driver, product_url=""):
    detail = empty_detail_fields()
    page_html = driver.page_source
    soup = BeautifulSoup(page_html, "html.parser")
    id_match = re.search(r"/(?:products|posts)/([a-zA-Z0-9-]+)", product_url or driver.current_url)
    detail["id"] = id_match.group(1) if id_match else ""
    meta_description = soup.select_one("meta[name='description']")
    if meta_description and meta_description.has_attr("content"):
        detail["description"] = meta_description["content"].strip()[:500]
    if not detail["description"]:
        desc_tag = soup.select_one("main p") or soup.select_one("article p") or soup.select_one("section p")
        if desc_tag:
            detail["description"] = desc_tag.get_text(" ", strip=True)[:500]
    detail["description_detaillee"] = detail["description"]
    review_link = soup.select_one("a[href*='/reviews']")
    if review_link:
        detail["reviews"] = review_link.get_text(" ", strip=True)
    else:
        text_blob = soup.get_text(" ", strip=True)
        review_match = re.search(r"(\d[\d,]*)\s+reviews?", text_blob, re.IGNORECASE)
        if review_match:
            detail["reviews"] = review_match.group(1)
    website_tag = None
    for anchor in soup.select("a[href^='http']"):
        href = anchor.get("href", "")
        if href and "producthunt.com" not in href:
            website_tag = anchor
            break
    if website_tag and website_tag.has_attr("href"):
        detail["website"] = website_tag["href"]
    maker_tags = soup.find_all("a", href=lambda href: href and ("/maker/" in href or "/makers/" in href))
    makers = []
    for tag in maker_tags:
        text = tag.get_text(" ", strip=True)
        if text and text not in makers:
            makers.append(text)
    detail["makers"] = ", ".join(makers[:6])
    detail["developers"] = detail["makers"]
    category_tags = soup.select("a[href^='/categories/']")
    categories = []
    for tag in category_tags:
        text = tag.get_text(" ", strip=True)
        if text and text not in categories:
            categories.append(text)
    detail["topics"] = ", ".join(categories[:10])
    detail["activity_types"] = detail["topics"]
    detail["app_type"] = categories[0] if categories else ""
    return detail


def click_browse_wait_extract_return(driver, wait, produits, search_url):
    all_products_data = []
    
    for i in range(len(produits)):
        nom_app = produits[i].get("name", f"product_{i + 1}")
        product_url = produits[i].get("product_url", "")
        
        if not product_url:
            print(f"Pas d'URL pour : {nom_app}")
            continue
        try:
            # --- ETAPE 1 : BROWSE (Naviguer vers la page produit dans un nouvel onglet) ---
            driver.execute_script(f"window.open('{product_url}', '_blank');")
            driver.switch_to.window(driver.window_handles[-1])
            # --- ETAPE 2 : WAIT (Attendre le chargement des donnees requises) ---
            wait.until(
                lambda d: len(
                    d.find_elements(By.CSS_SELECTOR, "main p, article p, section p, a[href*='/reviews'], a[href^='/categories/'], h1")
                ) > 0
            )
            # --- ETAPE 3 : EXTRACT (Extraire la description et les reviews) ---
            detail = extract_detail_fields(driver, product_url)
            description_complete = ""
            for selector in ["main p", "article p", "section p", "p"]:
                try:
                    el = driver.find_element(By.CSS_SELECTOR, selector)
                    description_complete = el.text.strip()
                    if description_complete:
                        break
                except Exception:
                    continue
            if description_complete:
                detail["description_detaillee"] = description_complete
                detail["description"] = description_complete[:500]
            avis_clients = detail.get("reviews", "")
            try:
                reviews_el = driver.find_element(By.CSS_SELECTOR, "a[href*='/reviews']")
                avis_clients = reviews_el.text.strip() or avis_clients
            except Exception:
                pass
            detail["reviews"] = avis_clients
            product_details = {
                **produits[i],
                **detail,
                "nom": nom_app,
            }
            all_products_data.append(product_details)
            print(f"Donnees extraites pour : {nom_app}")
        except TimeoutException:
            print(f"Timeout detail pour : {nom_app}")
        except Exception as error:
            print(f"Erreur detail pour {nom_app}: {error}")
        finally:
            # --- ETAPE 4 : RETURN (Fermer l'onglet et revenir a la page principale) ---
            try:
                if len(driver.window_handles) > 1:
                    driver.close()
                    driver.switch_to.window(driver.window_handles[0])
            except Exception as e:
                print(f"Erreur lors du retour a l'onglet principal: {e}")
                if len(driver.window_handles) > 0:
                    driver.switch_to.window(driver.window_handles[0])
    return all_products_data


def save_incremental(all_products, output_file=OUTPUT_JSON):
    json_file = "result_scrap_selenium2.json"
    with open(json_file, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)
    if all_products:
        csv_file = "result_scrap_selenium2.csv"
        with open(csv_file, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=all_products[0].keys())
            writer.writeheader()
            writer.writerows(all_products)


def create_driver():
    return webdriver.Firefox(options=options)


def scrape(start_page=START_PAGE, max_pages=MAX_PAGES, max_empty_pages=MAX_EMPTY_PAGES):
    all_products = []
    seen_names = set()
    page_number = start_page
    empty_pages = 0
    while page_number < start_page + max_pages and empty_pages < max_empty_pages:
        search_url = f"https://www.producthunt.com/search?q=mental+health+ai&page={page_number}"
        print(f"Scraping page {page_number} -> {search_url}")
        attempt = 0
        page_had_products = False
        while attempt < 3:
            attempt += 1
            driver = None
            try:
                driver = create_driver()
                wait = WebDriverWait(driver, 20)
                driver.get(search_url)
                wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "button[data-test^='spotlight-result-product-']")))
                time.sleep(2)
                products = parse_products_from_html(driver.page_source)
                if not products:
                    print(f"No products parsed on page {page_number} (attempt {attempt}).")
                    time.sleep(2)
                    continue
                page_had_products = True
                page_products = []
                for product in products:
                    key = product.get("name")
                    if not key or key in seen_names:
                        continue
                    seen_names.add(key)
                    product.update(empty_detail_fields())
                    product["row_id"] = len(all_products) + 1
                    page_products.append(product)
                details = click_browse_wait_extract_return(driver, wait, page_products, search_url)
                detail_map = {d.get("name") or d.get("nom"): d for d in details}
                for product in page_products:
                    product.update(detail_map.get(product.get("name"), {}))
                all_products.extend(page_products)
                print(f"Found {len(page_products)} products on page {page_number}")
                break
            except Exception as error:
                print(f"Driver error on page {page_number} attempt {attempt}: {type(error).__name__}: {error}")
                time.sleep(3 * attempt)
            finally:
                if driver is not None:
                    try:
                        driver.quit()
                    except Exception:
                        pass
        if page_had_products:
            empty_pages = 0
        else:
            empty_pages += 1
            print(f"Empty pages in a row: {empty_pages}/{max_empty_pages}")
        save_incremental(all_products, output_file=OUTPUT_JSON)
        page_number += 1
    print(f"Finished scraping. Total products: {len(all_products)}")
if __name__ == "__main__":
    max_pages = MAX_PAGES
    if len(sys.argv) >= 2:
        try:
            max_pages = int(sys.argv[1])
        except Exception:
            max_pages = MAX_PAGES
    scrape(start_page=START_PAGE, max_pages=max_pages)


Scraping page 1 -> https://www.producthunt.com/search?q=mental+health+ai&page=1
Pas d'URL pour : Mental
Pas d'URL pour : mentalport-app: AI Mental Health Coach
Pas d'URL pour : Yuna AI — Your Mental Health Coach
Pas d'URL pour : Ash
Pas d'URL pour : todai
Pas d'URL pour : TherapyAI
Pas d'URL pour : Unloop
Pas d'URL pour : MYND
Pas d'URL pour : Soula Care
Pas d'URL pour : Guin - Your GoodMind Buddy
Found 10 products on page 1
Scraping page 2 -> https://www.producthunt.com/search?q=mental+health+ai&page=2
Pas d'URL pour : MoodGallery: Emotions to art
Pas d'URL pour : Life Note
Pas d'URL pour : FitAction
Pas d'URL pour : Eve: Workplace Stress AI Coach
Pas d'URL pour : Psyche
Pas d'URL pour : Reset: Journal to reduce anxiety
Pas d'URL pour : Pallie
Pas d'URL pour : MindGuide
Pas d'URL pour : commitify.me
Pas d'URL pour : Anonymous Health
Found 10 products on page 2
Scraping page 3 -> https://www.producthunt.com/search?q=mental+health+ai&page=3
Pas d'URL pour : Mentat Ai - Your Mental Healt

In [9]:
import json
from google_play_scraper import search, app as get_app_details, reviews, Sort

"""COMMENTAIRES : Ce script extrait les données de l'API Google Play Store.
En commençant par la recherche des applications qui correspondent à un certain mot-clé.
Puis, on extrait les commentaires de chaque application.
Ensuite, on enregistre les données dans un fichier JSON.
"""

def extract_mental_health_data():
    print("Démarrage de l'extraction des données via Google Play API...")
    
    # On cherche des mots-clés spécifiques
    search_results = search("AI mental health wellness", lang="en", country="us")
    
    # On évite le conflit de nom et on extrait correctement l'appId via dictionnaire
    app_ids = []
    # On prend les 100 premiers ou le max disponible si moins de 100
    limite = min(100, len(search_results))
    
    for i in range(limite):
        app_item = search_results[i]
        app_ids.append(app_item.get("appId"))
        
    print(f"{len(app_ids)} applications identifiées pour l'analyse.")

    all_apps_data = []

    # Boucle d'extraction pour chaque application
    for app_id in app_ids:
        if not app_id:
            continue
            
        print(f"Extraction en cours pour : {app_id}...")
        
        try:
            # Utilisation de get_app_details (renommé dans l'import pour plus de clarté)
            app_details = get_app_details(
                app_id,
                lang='en', 
                country='us' 
            )
            
            # Récupération des avis (reviews)
            app_reviews, _ = reviews(
                app_id,
                lang='en',
                country='us',
                sort=Sort.MOST_RELEVANT, 
                count=100 
            )
            
            # Structuration de la donnée
            structured_data = {
                "id_application": app_details.get("appId"),
                "nom_application": app_details.get("title"),
                "developpeur": app_details.get("developer"),
                "note_globale": app_details.get("score"),
                "telechargements": app_details.get("installs"),
                "description_complete": app_details.get("description"),
                "prix": app_details.get("price"),
                "genre": app_details.get("genre"),
                
                "avis_utilisateurs": [
                    {
                        "utilisateur": review.get("userName"),
                        "note": review.get("score"),
                        # Utilisation de .get() pour éviter les KeyErrors sur les dates ou contenus vides
                        "date": review.get("at").strftime("%Y-%m-%d %H:%M:%S") if review.get("at") else None,
                        "commentaire": review.get("content"),
                        "pouces_utiles": review.get("thumbsUpCount", 0)
                    }
                    for review in app_reviews
                ]
            }
            
            all_apps_data.append(structured_data)
            
        except Exception as e:
            print(f"Erreur lors de l'extraction pour {app_id}: {e}")

    # Sauvegarde dans un fichier JSON
    print("Sauvegarde des données dans 'mental_health_ai_apps.json'...")
    with open("mental_health_ai_apps.json", "w", encoding="utf-8") as f:
        json.dump(all_apps_data, f, indent=4, ensure_ascii=False)
        
    print("Opération terminée avec succès !")

# Exécution du script
if __name__ == "__main__":
    extract_mental_health_data()

Démarrage de l'extraction des données via Google Play API...
11 applications identifiées pour l'analyse.
Extraction en cours pour : bot.touchkin...
Extraction en cours pour : xyz.slingshot.ashley.app...
Extraction en cours pour : com.cbt.mindhealthy...
Extraction en cours pour : com.wave6.onsen...
Extraction en cours pour : com.getmental.mental...
Extraction en cours pour : com.vos.chatmind.mentalhealth.therapy.coach...
Extraction en cours pour : com.ai.mentat...
Extraction en cours pour : com.wellness.ai...
Extraction en cours pour : com.healthmind.app...
Extraction en cours pour : com.elomia.health...
Extraction en cours pour : com.capitalx.blissfully...
Sauvegarde des données dans 'mental_health_ai_apps.json'...
Opération terminée avec succès !


In [10]:
# get the programming language in result_project and parse it to build a visualisation with streamlit